# 🚀 VentureLens AI 2.0 — Open Model Fine-Tuning Guide (LoRA / PEFT)

This notebook demonstrates how to fine-tune an open foundation model (e.g., `meta-llama/Llama-3.2-3B-Instruct`, `google/gemma-2-2b-it`, or `Qwen/Qwen2.5-7B-Instruct`) on your custom **VentureLens Startup Evaluation Dataset** for free using **Google Colab (T4 GPU)**, **Hugging Face Transformers**, **PEFT (LoRA)**, and **TRL (SFTTrainer)**.

---

## 1. Install Prerequisites & Hugging Face Libraries

In [ ]:
!pip install -q -U torch transformers datasets peft trl bitsandbytes accelerate

## 2. Load VentureLens Evaluation Dataset (JSONL)

In [ ]:
from datasets import load_dataset

# Upload your dataset JSONL generated by scripts/export-fine-tuning-dataset.mjs
dataset = load_dataset('json', data_files='venturelens_fine_tuning_dataset.jsonl')
print('Dataset sample:', dataset['train'][0])

## 3. Load Model with 4-Bit Quantization (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded successfully with QLoRA 4-bit quantization.")

## 4. Configure LoRA (Low-Rank Adaptation)

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Train Model with Hugging Face TRL SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./venturelens_lora_model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=100,
    fp16=True,
    save_strategy="steps",
    save_steps=50
)

def formatting_prompts_func(example):
    output_texts = []
    for i in range(len(example['instruction'])):
        text = f"### System: {example['instruction'][i]}\n\n### Input: {example['input'][i]}\n\n### Response: {example['output'][i]}"
        output_texts.append(text)
    return output_texts

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    peft_config=peft_config,
    formatting_func=formatting_prompts_func,
    args=training_args
)

trainer.train()
print("Training completed!")

## 6. Save & Push Adapter to Hugging Face Hub

In [ ]:
model.save_pretrained("./venturelens_adapter")
tokenizer.save_pretrained("./venturelens_adapter")
# Option to push to Hugging Face Hub:
# model.push_to_hub("your-username/venturelens-llama3.2-3b-lora")
print("VentureLens LoRA Adapter saved successfully.")